# Yelp Data Exploration

[<img  alt="image" src="images/banner.png">](https://public.tableau.com/app/profile/christopher.magno/viz/YelpAnalysis_17653606827830/YelpAnalysis)

For this project, I selected a dataset composed of relational tables connected through primary and foreign keys. The
goal was to focus on data wrangling, exploratory analysis, and visualization in Tableau. The data comes from the Yelp
Open Dataset, which is available [here](https://business.yelp.com/data/resources/open-dataset/). The dataset is provided
across five separate JSON files.

| Files                               | Shape (row x columns) | Size     |
|-------------------------------------|-----------------------|----------|
| yelp_academic_dataset_business.json | 150,346 x 60          | 118.9 MB |
| yelp_academic_dataset_checkin.json  | 131,930 x 2           | 287 MB   |
| yelp_academic_dataset_review.json   | 6,990,280 x 9         | 5.34 GB  |
| yelp_academic_dataset_tip.json      | 908,915 x 5           | 180.6 MB |
| yelp_academic_dataset_user.json     | 1,987,897 x 22        | 3.36 GB  |

## Dataset Modeling
<img width="1000" alt="image" src="images/img1.png">


## The goal of this project
* Get total businesses, total review count, and average rating for the regions provided per category overtime (spark lines)
* Get total business, total review count, and average rating per price tiers ($, $$, $$$, $$$$)
* Get user engagement by hour/day of week by checkins
* Get top categories
* Get top 10 businesses (per region)
* Evaluate market share distribution to determine which segment dominates


## Stretch goals
* Sentiment Analysis and word cloud or negative and positive keywords on the 6.9M rows of customer reviews and tips
  * Sentiment trends over time


## ⚒️ Tools Used
* Python
  * Pandas
* Tableau


To view the Tableau visualization, please click [here](https://public.tableau.com/app/profile/christopher.magno/viz/YelpAnalysis_17653606827830/YelpAnalysis)
<img width="1000" alt="image" src="images/img2.png">

In [2]:
import importlib
import datetime
import json
from pathlib import Path

import pandas as pd
import numpy as np

import lib
importlib.reload(lib)

PROJECT_DIRECTORY = lib.get_project_dir()
PROJECT_NAME = lib.get_project_name()
log = lib.getLogger(PROJECT_NAME)

EXPORT_DIRECTORY = PROJECT_DIRECTORY / 'clean'

In [2]:
path_dataset_business = '/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/yelp_academic_dataset_business.json'  # 118.9 MB

In [3]:
df_business = lib.read_json(path_dataset_business)

In [4]:
df_business.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 60 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   business_id                            150346 non-null  object 
 1   name                                   150346 non-null  object 
 2   address                                150346 non-null  object 
 3   city                                   150346 non-null  object 
 4   state                                  150346 non-null  object 
 5   postal_code                            150346 non-null  object 
 6   latitude                               150346 non-null  float64
 7   longitude                              150346 non-null  float64
 8   stars                                  150346 non-null  float64
 9   review_count                           150346 non-null  int64  
 10  is_open                                150346 non-null  

In [5]:
df_business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,attributes.AcceptsInsurance,attributes.BestNights,attributes.BYOB,attributes.Corkage,attributes.BYOBCorkage,attributes.HairSpecializesIn,attributes.Open24Hours,attributes.RestaurantsCounterService,attributes.AgesAllowed,attributes.DietaryRestrictions
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df_business_cp1 = df_business.copy()

### Creating backup

In [7]:
df_business_cp1 = df_business.copy()

### Looking for Risky columns to potentially drop

In [8]:
risky_columns = lib.risky_columns(df_business_cp1, err_rate=.9)
risky_columns

,error_percent,error_count
hours,100.00%,150346
attributes.CoatCheck,96.29%,144762
attributes.DriveThru,94.84%,142586
attributes,100.00%,150346
attributes.Smoking,96.96%,145779
attributes.Music,95.00%,142825
attributes.GoodForDancing,96.92%,145718
attributes.AcceptsInsurance,96.20%,144633
attributes.BestNights,96.21%,144652
attributes.BYOB,97.04%,145895


In [9]:
df_business_cp1 = df_business_cp1.drop(risky_columns.index, axis=1)
df_business_cp1.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'categories', 'attributes.ByAppointmentOnly',
       'attributes.BusinessAcceptsCreditCards', 'hours.Monday',
       'hours.Tuesday', 'hours.Wednesday', 'hours.Thursday', 'hours.Friday',
       'hours.Saturday', 'attributes.BikeParking',
       'attributes.RestaurantsPriceRange2', 'attributes.RestaurantsTakeOut',
       'attributes.RestaurantsDelivery', 'attributes.Caters',
       'attributes.WiFi', 'attributes.BusinessParking',
       'attributes.WheelchairAccessible', 'attributes.HappyHour',
       'attributes.OutdoorSeating', 'attributes.HasTV',
       'attributes.RestaurantsReservations', 'attributes.DogsAllowed',
       'hours.Sunday', 'attributes.Alcohol', 'attributes.GoodForKids',
       'attributes.RestaurantsAttire', 'attributes.Ambience',
       'attributes.RestaurantsTableService',
       'attributes.RestaurantsGoodForGroups', 'a

### Creating backup

In [10]:
df_business_cp2 = df_business_cp1.copy()

### Fixing Categories
Going to store into multiple datasets (categories table, business/categories table [build relationship between the two])

In [11]:
df_business_cp2['categories'].value_counts()

categories
Beauty & Spas, Nail Salons                                                                                       1012
Restaurants, Pizza                                                                                                935
Nail Salons, Beauty & Spas                                                                                        934
Pizza, Restaurants                                                                                                823
Restaurants, Mexican                                                                                              728
                                                                                                                 ... 
Dermatologists, Health & Medical, Cosmetic Surgeons, Doctors, Acne Treatment, Skin Care, Beauty & Spas              1
Home Services, Home & Garden, Nurseries & Gardening, Hardware Stores, Shopping, Building Supplies, Appliances       1
Food Trucks, Smokehouse, Restaurants, Food, B

Building out the data for the categories

In [12]:
# Holds data per index/business_id/associated categories
categories_dict = {}
# Holds all categories
categories_set = set()
for idx, row in df_business_cp2.iterrows():
    if row['categories']:
        categories_dict[idx] = {'business_id': row['business_id'], 'categories': row['categories'].split(', ')}
    else:
        categories_dict[idx] = {'business_id': row['business_id'], 'categories': []}

    # Update the full list of categories
    categories_set.update(categories_dict[idx]['categories'])

categories_set = sorted(list(categories_set))
categories_dict

{0: {'business_id': 'Pns2l4eNsfO8kk83dixA6A',
  'categories': ['Doctors',
   'Traditional Chinese Medicine',
   'Naturopathic/Holistic',
   'Acupuncture',
   'Health & Medical',
   'Nutritionists']},
 1: {'business_id': 'mpf3x-BjTdTEA3yCZrAYPw',
  'categories': ['Shipping Centers',
   'Local Services',
   'Notaries',
   'Mailbox Centers',
   'Printing Services']},
 2: {'business_id': 'tUFrWirKiKi_TAnsVWINQQ',
  'categories': ['Department Stores',
   'Shopping',
   'Fashion',
   'Home & Garden',
   'Electronics',
   'Furniture Stores']},
 3: {'business_id': 'MTSW4McQd7CbVtyjqoe9mw',
  'categories': ['Restaurants',
   'Food',
   'Bubble Tea',
   'Coffee & Tea',
   'Bakeries']},
 4: {'business_id': 'mWMc6_wTdE0EUBKIGXDVfA',
  'categories': ['Brewpubs', 'Breweries', 'Food']},
 5: {'business_id': 'CF33F8-E6oudUQ46HnavjQ',
  'categories': ['Burgers',
   'Fast Food',
   'Sandwiches',
   'Food',
   'Ice Cream & Frozen Yogurt',
   'Restaurants']},
 6: {'business_id': 'n_0UpQx1hsNbnPUSlodU8w',
 

In [13]:
categories_set

['& Probates',
 '3D Printing',
 'ATV Rentals/Tours',
 'Acai Bowls',
 'Accessories',
 'Accountants',
 'Acne Treatment',
 'Active Life',
 'Acupuncture',
 'Addiction Medicine',
 'Adoption Services',
 'Adult',
 'Adult Education',
 'Adult Entertainment',
 'Advertising',
 'Aerial Fitness',
 'Aerial Tours',
 'Aestheticians',
 'Afghan',
 'African',
 'Air Duct Cleaning',
 'Aircraft Dealers',
 'Aircraft Repairs',
 'Airlines',
 'Airport Lounges',
 'Airport Shuttles',
 'Airport Terminals',
 'Airports',
 'Airsoft',
 'Allergists',
 'Alternative Medicine',
 'Amateur Sports Teams',
 'American (New)',
 'American (Traditional)',
 'Amusement Parks',
 'Anesthesiologists',
 'Animal Assisted Therapy',
 'Animal Physical Therapy',
 'Animal Shelters',
 'Antiques',
 'Apartment Agents',
 'Apartments',
 'Appliances',
 'Appliances & Repair',
 'Appraisal Services',
 'Aquarium Services',
 'Aquariums',
 'Arabic',
 'Arcades',
 'Archery',
 'Architects',
 'Architectural Tours',
 'Argentine',
 'Armenian',
 'Art Classes',

#### Creating the categories dataset

In [14]:
df_business_categories_id = pd.DataFrame(enumerate(categories_set), columns=['category_id', 'category_name']).set_index('category_id')
df_business_categories_id

,category_name
category_id,
0,& Probates
1,3D Printing
2,ATV Rentals/Tours
3,Acai Bowls
4,Accessories
...,...
1306,Wraps
1307,Yelp Events
1308,Yoga


#### [DONT RUN THIS SECTION] Grouping categories though ML (these are actually more like tags...)
- Food & Dining
- Shopping
- Beauty / Personal Care
- Health & Wellness
- Professional Services
- Home Services
- Automotive
- Fitness
- Pets
- Entertainment
- Financial Services
- Real Estate

##### Build the embeddings model
##### TODO: Need to refine the grouping or take care of some edge cases

In [15]:
group_name_mapping = {
    0: 'Pets',
    1: 'Shopping',
    2: 'Professional Services',
    3: 'Beauty / Personal Care',
    4: 'Education',
    5: 'Food & Dining',
    6: 'Financial Services',
    7: 'Entertainment',
    8: 'Fitness',
    9: 'Health & Wellness',
    10: 'Home Services',
    11: 'Transportation',
}

In [16]:
from sentence_transformers import SentenceTransformer

model_name = 'all-MiniLM-L6-v2'
# model_name = 'multi-qa-mpnet-base-dot-v1'
model = SentenceTransformer(model_name)
embeddings = model.encode(df_business_categories_id['category_name'], show_progress_bar=True)
# df_business_categories_id.to_pickle(export_clean_directory / 'categories_with_embeddings.pkl')

/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
sentence_transformers.SentenceTransformer: INFO: Use pytorch device_name: mps
sentence_transformers.SentenceTransformer: INFO: Load pretrained SentenceTransformer: all-MiniLM-L6-v2
Batches: 100%|██████████| 41/41 [00:00<00:00, 56.22it/s]


##### With hdbscan

In [17]:
# import hdbscan
# x = np.vstack(embeddings.tolist())
# clusterer = hdbscan.HDBSCAN(min_cluster_size=3, min_samples=3, metric='euclidean')
# df_business_categories_id['group_id'] = clusterer.fit_predict(x)
# len(df_business_categories_id['group_id'].value_counts())

##### With sklean clustering

In [18]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

embeddings = normalize(embeddings)
num_clusters = 12

kmeans = KMeans(
    n_clusters=num_clusters,
    random_state=2,
    n_init='auto',
    max_iter=500,
)
group_ids = kmeans.fit_predict(embeddings)
group_ids

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


array([ 6,  2, 11, ...,  8,  2,  0], shape=(1311,), dtype=int32)

##### Create DataFrame for groups

In [19]:
df_business_groups_id = pd.DataFrame(
    {'group_id': group_name_mapping.keys(),
     'group_name': group_name_mapping.values()}
).set_index('group_id')
df_business_groups_id

,group_name
group_id,
0,Pets
1,Shopping
2,Professional Services
3,Beauty / Personal Care
4,Education
5,Food & Dining
6,Financial Services
7,Entertainment
8,Fitness


#### Creating the business/categories key dataset

In [20]:
business_categories_data = []
for idx, business_data in categories_dict.items():
    business_id = business_data['business_id']
    for category in business_data['categories']:
        category_id = categories_set.index(category)
        business_categories_data.append({'business_id': business_id, 'category_id': category_id})
business_categories_data

[{'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_id': 359},
 {'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_id': 1200},
 {'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_id': 798},
 {'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_id': 8},
 {'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_id': 551},
 {'business_id': 'Pns2l4eNsfO8kk83dixA6A', 'category_id': 813},
 {'business_id': 'mpf3x-BjTdTEA3yCZrAYPw', 'category_id': 1065},
 {'business_id': 'mpf3x-BjTdTEA3yCZrAYPw', 'category_id': 712},
 {'business_id': 'mpf3x-BjTdTEA3yCZrAYPw', 'category_id': 808},
 {'business_id': 'mpf3x-BjTdTEA3yCZrAYPw', 'category_id': 721},
 {'business_id': 'mpf3x-BjTdTEA3yCZrAYPw', 'category_id': 951},
 {'business_id': 'tUFrWirKiKi_TAnsVWINQQ', 'category_id': 337},
 {'business_id': 'tUFrWirKiKi_TAnsVWINQQ', 'category_id': 1069},
 {'business_id': 'tUFrWirKiKi_TAnsVWINQQ', 'category_id': 434},
 {'business_id': 'tUFrWirKiKi_TAnsVWINQQ', 'category_id': 572},
 {'business_id': 'tUFrWirKiKi_TAnsVWINQ

In [21]:
df_business_categories_key = pd.DataFrame(business_categories_data)
df_business_categories_key

,business_id,category_id
0,Pns2l4eNsfO8kk83dixA6A,359
1,Pns2l4eNsfO8kk83dixA6A,1200
2,Pns2l4eNsfO8kk83dixA6A,798
3,Pns2l4eNsfO8kk83dixA6A,8
4,Pns2l4eNsfO8kk83dixA6A,551
...,...,...
668587,mtGm22y5c2UHNXDFAjaPNw,135
668588,jV_XOycEzSlTx-65W906pg,115
668589,jV_XOycEzSlTx-65W906pg,880
668590,jV_XOycEzSlTx-65W906pg,917


In [22]:
sorted_group_ids = []
for idx, row in df_business_categories_id.iterrows():
    sorted_group_ids.append(int(group_ids[idx]))
sorted_group_ids

[6,
 2,
 11,
 3,
 2,
 6,
 2,
 8,
 9,
 9,
 6,
 5,
 4,
 7,
 7,
 8,
 7,
 2,
 5,
 5,
 10,
 1,
 10,
 8,
 11,
 11,
 11,
 11,
 8,
 9,
 9,
 8,
 5,
 5,
 11,
 9,
 0,
 0,
 0,
 2,
 6,
 11,
 3,
 10,
 6,
 6,
 8,
 5,
 8,
 8,
 6,
 7,
 5,
 5,
 4,
 7,
 7,
 7,
 7,
 2,
 4,
 7,
 1,
 7,
 8,
 2,
 7,
 5,
 11,
 2,
 11,
 1,
 7,
 9,
 5,
 5,
 2,
 10,
 10,
 6,
 6,
 10,
 10,
 6,
 2,
 6,
 6,
 2,
 8,
 9,
 2,
 6,
 8,
 3,
 6,
 3,
 6,
 5,
 6,
 6,
 3,
 3,
 2,
 4,
 3,
 3,
 4,
 8,
 8,
 5,
 1,
 8,
 3,
 11,
 8,
 11,
 3,
 3,
 3,
 11,
 11,
 7,
 4,
 5,
 2,
 1,
 8,
 8,
 11,
 11,
 10,
 10,
 8,
 1,
 7,
 8,
 6,
 11,
 10,
 1,
 3,
 9,
 6,
 8,
 1,
 1,
 10,
 7,
 8,
 8,
 2,
 1,
 2,
 6,
 7,
 1,
 11,
 11,
 2,
 11,
 8,
 8,
 3,
 5,
 8,
 3,
 3,
 3,
 3,
 2,
 5,
 8,
 3,
 11,
 3,
 1,
 3,
 5,
 11,
 11,
 7,
 11,
 6,
 6,
 6,
 3,
 4,
 6,
 7,
 6,
 11,
 3,
 5,
 5,
 2,
 5,
 11,
 5,
 1,
 1,
 11,
 11,
 3,
 7,
 5,
 6,
 6,
 1,
 1,
 6,
 11,
 6,
 10,
 2,
 2,
 4,
 9,
 2,
 4,
 5,
 2,
 7,
 2,
 2,
 2,
 2,
 2,
 7,
 3,
 2,
 4,
 3,
 6,
 8,
 1,
 3,
 3,
 1,
 3,
 6,


#### Exporting the DataFrames

In [23]:
df_business_categories_id['group_id'] = sorted_group_ids
df_business_categories_id

,category_name,group_id
category_id,,
0,& Probates,6
1,3D Printing,2
2,ATV Rentals/Tours,11
3,Acai Bowls,3
4,Accessories,2
...,...,...
1306,Wraps,2
1307,Yelp Events,7
1308,Yoga,8


In [24]:
lib.export_data(EXPORT_DIRECTORY / 'yelp_business_groups_id.csv', df_business_groups_id)
lib.export_data(EXPORT_DIRECTORY / 'yelp_business_categories_id.csv', df_business_categories_id)
lib.export_data(EXPORT_DIRECTORY / 'yelp_business_categories_key.csv', df_business_categories_key)

PortfolioLogger.lib.tools: INFO: Exporting 12 elements (0.00 MB) to /Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_business_groups_id_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10f9a1940> took 0.003 secs to complete.
PortfolioLogger.lib.tools: INFO: Exporting 2622 elements (0.13 MB) to /Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_business_categories_id_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10f9a1940> took 0.002 secs to complete.
PortfolioLogger.lib.tools: INFO: Exporting 1337184 elements (50.37 MB) to /Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_business_categories_key_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10f9a1940> took 0.364 secs to complete.


PosixPath('/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_business_categories_key_CLEAN.csv')

### Creating backup

In [25]:
df_business_cp3 = df_business_cp2.copy()

### Cleaning Hours columns
Going to create a new DataFrame for these columns

In [26]:
df_business_cp3['hours.Sunday']

0               NaN
1               NaN
2          8:0-22:0
3          7:0-21:0
4         12:0-18:0
            ...    
150341    11:0-17:0
150342     0:0-16:0
150343          NaN
150344    10:0-17:0
150345          NaN
Name: hours.Sunday, Length: 150346, dtype: object

In [27]:
def convert_unit(u):
    timestamp = [int(item) for item in u.split(':')]
    t = datetime.time(*timestamp)
    return t.strftime('%H:%M:%S')

def convert_time(t):
    start, end = t.split('-')
    start = convert_unit(start)
    end = convert_unit(end)
    return start, end

Creating DataFrame

In [28]:
days = {
    'Sunday': 0,
    'Monday': 1,
    'Tuesday': 2,
    'Wednesday': 3,
    'Thursday': 4,
    'Friday': 5,
    'Saturday': 6
}
days_mapping = {
    'days_id': [0,1,2,3,4,5,6],
    'day_of_week': ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
}
days

{'Sunday': 0,
 'Monday': 1,
 'Tuesday': 2,
 'Wednesday': 3,
 'Thursday': 4,
 'Friday': 5,
 'Saturday': 6}

In [29]:
hours_data = {
    'business_id': [],
    'day_of_week': [],
    'open_time': [],
    'close_time': [],
    'is_closed': []
}
to_drop =set()

for col, series in df_business_cp3.items():
    if col.startswith('hours.'):
        to_drop.add(col)
        day = col.replace('hours.', '')
        print(day)
        for idx, item in series.items():
            business_id = df_business_cp3.loc[idx, 'business_id']
            hours_data['business_id'].append(business_id)
            hours_data['day_of_week'].append(days[day])
            try:
                if np.isnan(item) or pd.isna(item):
                    hours_data['open_time'].append(pd.NA)
                    hours_data['close_time'].append(pd.NA)
                    hours_data['is_closed'].append(True)
            except TypeError:
                try:
                    open_time, close_time = convert_time(item)
                    hours_data['open_time'].append(open_time)
                    hours_data['close_time'].append(close_time)
                    hours_data['is_closed'].append(False)
                except:
                    log.error(f'Error in parsing {idx}: ({business_id})')

hours_data

Monday
Tuesday
Wednesday
Thursday
Friday
Saturday
Sunday


{'business_id': ['Pns2l4eNsfO8kk83dixA6A',
  'mpf3x-BjTdTEA3yCZrAYPw',
  'tUFrWirKiKi_TAnsVWINQQ',
  'MTSW4McQd7CbVtyjqoe9mw',
  'mWMc6_wTdE0EUBKIGXDVfA',
  'CF33F8-E6oudUQ46HnavjQ',
  'n_0UpQx1hsNbnPUSlodU8w',
  'qkRM_2X51Yqxk3btlwAQIg',
  'k0hlBqXX-Bt0vf1op7Jr1w',
  'bBDDEgkFA1Otx9Lfe7BZUQ',
  'UJsufbvfyfONHeWdvAHKjA',
  'eEOYSgkmpB90uNA7lDOMRA',
  'il_Ro8jwPlHresjw9EGmBg',
  'jaxMSoInw8Poo3XeMJt8lQ',
  '0bPLkL0QhhPO5kt1_EXmNQ',
  'MUTTqe8uqyMdBl186RmNeA',
  'rBmpy_Y1UbBx8ggHlyb7hA',
  'M0XSSHqrASOnhgbWDJIpQA',
  '8wGISYjYkE2tSqn3cDMu8A',
  'ROeacJQwBeh05Rqg7F6TCg',
  'WKMJwqnfZKsAae75RMP6jA',
  'qhDdDeI3K4jy2KyzwFN53w',
  'kfNv-JZpuN6TVNSO6hHdkw',
  '9OG5YkX1g2GReZM0AskizA',
  '4iRzR7OaS-QaSXuvYxEGKA',
  'PSo_C1Sfa13JHjzVNW6ziQ',
  'noByYNtDLQAra9ccqxdfDw',
  'tMkwHmWFUEXrC9ZduonpTg',
  'QdN72BWoyFypdGJhhI5r7g',
  'sqSqqLy0sN8n2IZrAbzidQ',
  'fvWn8oXXwbj2l79cochZyw',
  'Mjboz24M9NlBeiOJKLEd_Q',
  '8sshLb4UU7emeUDvtJWnpA',
  'kV_Q1oqis8Qli8dUoGpTyQ',
  'w_AMNoI1iG9eay7ncmc67w',
  'aP

In [30]:
hours_data['day_of_week']

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


In [31]:
to_drop

{'hours.Friday',
 'hours.Monday',
 'hours.Saturday',
 'hours.Sunday',
 'hours.Thursday',
 'hours.Tuesday',
 'hours.Wednesday'}

#### Creating hours DataFrame

In [32]:
df_business_hours = pd.DataFrame(hours_data, columns=hours_data.keys())  # Make keys as index
df_business_hours

,business_id,day_of_week,open_time,close_time,is_closed
0,Pns2l4eNsfO8kk83dixA6A,1,<NA>,<NA>,True
1,mpf3x-BjTdTEA3yCZrAYPw,1,00:00:00,00:00:00,False
2,tUFrWirKiKi_TAnsVWINQQ,1,08:00:00,22:00:00,False
3,MTSW4McQd7CbVtyjqoe9mw,1,07:00:00,20:00:00,False
4,mWMc6_wTdE0EUBKIGXDVfA,1,<NA>,<NA>,True
...,...,...,...,...,...
1052417,IUQopTMmYQG-qRtBk-8QnA,0,11:00:00,17:00:00,False
1052418,c8GjPIOTGVmIemT7j5_SyQ,0,00:00:00,16:00:00,False
1052419,_QAMST-NrQobXduilWEqSw,0,<NA>,<NA>,True
1052420,mtGm22y5c2UHNXDFAjaPNw,0,10:00:00,17:00:00,False


#### Exporting Hours DataFrame

In [33]:
lib.export_data(EXPORT_DIRECTORY / 'yelp_business_hours.csv', df_business_hours)

PortfolioLogger.lib.tools: INFO: Exporting 5262110 elements (194.23 MB) to /Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_business_hours_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10f9a1940> took 1.069 secs to complete.


PosixPath('/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_business_hours_CLEAN.csv')

#### Dropping hours columns

In [34]:
df_business_cp3 = df_business_cp3.drop(to_drop, axis=1)

In [35]:
df_business_cp3.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,attributes.DogsAllowed,attributes.Alcohol,attributes.GoodForKids,attributes.RestaurantsAttire,attributes.Ambience,attributes.RestaurantsTableService,attributes.RestaurantsGoodForGroups,attributes.NoiseLevel,attributes.GoodForMeal,attributes.BusinessAcceptsBitcoin
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,...,NaN,u'none',NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,...,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Creating backup

In [36]:
df_business_cp4 = df_business_cp3.copy()

### Cleaning Attributes columns

In [37]:
not_risky = []
for day, series in df_business_cp4.items():
    if day.startswith('attributes.'):
        log.info(day)
        if not lib.isrisky(series, err_rate=.5):
            not_risky.append(day)
not_risky

PortfolioLogger.yelp_dataset: INFO: attributes.ByAppointmentOnly
PortfolioLogger.yelp_dataset: INFO: attributes.BusinessAcceptsCreditCards
PortfolioLogger.yelp_dataset: INFO: attributes.BikeParking
PortfolioLogger.yelp_dataset: INFO: attributes.RestaurantsPriceRange2
PortfolioLogger.yelp_dataset: INFO: attributes.RestaurantsTakeOut
PortfolioLogger.yelp_dataset: INFO: attributes.RestaurantsDelivery
PortfolioLogger.yelp_dataset: INFO: attributes.Caters
PortfolioLogger.yelp_dataset: INFO: attributes.WiFi
PortfolioLogger.yelp_dataset: INFO: attributes.BusinessParking
PortfolioLogger.yelp_dataset: INFO: attributes.WheelchairAccessible
PortfolioLogger.yelp_dataset: INFO: attributes.HappyHour
PortfolioLogger.yelp_dataset: INFO: attributes.OutdoorSeating
PortfolioLogger.yelp_dataset: INFO: attributes.HasTV
PortfolioLogger.yelp_dataset: INFO: attributes.RestaurantsReservations
PortfolioLogger.yelp_dataset: INFO: attributes.DogsAllowed
PortfolioLogger.yelp_dataset: INFO: attributes.Alcohol
Portf

['attributes.BusinessAcceptsCreditCards',
 'attributes.RestaurantsPriceRange2',
 'attributes.BusinessParking']

#### The attributes columns are all a little risky but I'll include them for extra data that shouldn't be relied on but can be helpful to see in a business

In [38]:
attributes_cols = []
for col, series in df_business_cp4.items():
    if col.startswith('attributes.'):
        attributes_cols.append(col)

lib.error_rates(df_business_cp4[attributes_cols])

,error_percent,error_count
attributes.ByAppointmentOnly,71.84%,108007
attributes.BusinessAcceptsCreditCards,20.34%,30581
attributes.BikeParking,51.69%,77708
attributes.RestaurantsPriceRange2,43.25%,65032
attributes.RestaurantsTakeOut,60.19%,90489
attributes.RestaurantsDelivery,62.57%,94064
attributes.Caters,73.31%,110219
attributes.WiFi,62.14%,93432
attributes.BusinessParking,39.42%,59261
attributes.WheelchairAccessible,80.74%,121393


In [39]:
df_business_cp4[attributes_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 24 columns):
 #   Column                                 Non-Null Count   Dtype 
---  ------                                 --------------   ----- 
 0   attributes.ByAppointmentOnly           42339 non-null   object
 1   attributes.BusinessAcceptsCreditCards  119765 non-null  object
 2   attributes.BikeParking                 72638 non-null   object
 3   attributes.RestaurantsPriceRange2      85314 non-null   object
 4   attributes.RestaurantsTakeOut          59857 non-null   object
 5   attributes.RestaurantsDelivery         56282 non-null   object
 6   attributes.Caters                      40127 non-null   object
 7   attributes.WiFi                        56914 non-null   object
 8   attributes.BusinessParking             91085 non-null   object
 9   attributes.WheelchairAccessible        28953 non-null   object
 10  attributes.HappyHour                   15171 non-null   object
 11  

In [40]:
df_business_cp4[attributes_cols]

,attributes.ByAppointmentOnly,attributes.BusinessAcceptsCreditCards,attributes.BikeParking,attributes.RestaurantsPriceRange2,attributes.RestaurantsTakeOut,attributes.RestaurantsDelivery,attributes.Caters,attributes.WiFi,attributes.BusinessParking,attributes.WheelchairAccessible,...,attributes.DogsAllowed,attributes.Alcohol,attributes.GoodForKids,attributes.RestaurantsAttire,attributes.Ambience,attributes.RestaurantsTableService,attributes.RestaurantsGoodForGroups,attributes.NoiseLevel,attributes.GoodForMeal,attributes.BusinessAcceptsBitcoin
0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,False,True,True,2,False,False,False,u'no',"{'garage': False, 'street': False, 'validated'...",True,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,False,False,True,1,True,False,True,u'free',"{'garage': False, 'street': True, 'validated':...",NaN,...,NaN,u'none',NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,True,True,NaN,True,NaN,False,NaN,"{'garage': None, 'street': None, 'validated': ...",True,...,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150341,False,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150342,NaN,True,True,2,NaN,NaN,NaN,u'no',"{'garage': False, 'street': False, 'validated'...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150343,NaN,True,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150344,NaN,True,True,4,None,None,NaN,NaN,"{'garage': False, 'street': False, 'validated'...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
for col, series in df_business_cp4.items():
    if col.startswith('attributes.'):
        log.info(col)
        log.info(df_business_cp4[col].unique())

PortfolioLogger.yelp_dataset: INFO: attributes.ByAppointmentOnly
PortfolioLogger.yelp_dataset: INFO: ['True' nan 'False' 'None']
PortfolioLogger.yelp_dataset: INFO: attributes.BusinessAcceptsCreditCards
PortfolioLogger.yelp_dataset: INFO: [nan 'True' 'False' 'None']
PortfolioLogger.yelp_dataset: INFO: attributes.BikeParking
PortfolioLogger.yelp_dataset: INFO: [nan 'True' 'False' 'None']
PortfolioLogger.yelp_dataset: INFO: attributes.RestaurantsPriceRange2
PortfolioLogger.yelp_dataset: INFO: [nan '2' '1' '3' '4' 'None']
PortfolioLogger.yelp_dataset: INFO: attributes.RestaurantsTakeOut
PortfolioLogger.yelp_dataset: INFO: [nan 'False' 'True' 'None']
PortfolioLogger.yelp_dataset: INFO: attributes.RestaurantsDelivery
PortfolioLogger.yelp_dataset: INFO: [nan 'False' 'True' 'None']
PortfolioLogger.yelp_dataset: INFO: attributes.Caters
PortfolioLogger.yelp_dataset: INFO: [nan 'False' 'True' 'None']
PortfolioLogger.yelp_dataset: INFO: attributes.WiFi
PortfolioLogger.yelp_dataset: INFO: [nan "u'

In [42]:
special_attribute = 'attributes.RestaurantsPriceRange2'
bool_attributes = [
    'attributes.ByAppointmentOnly',
    'attributes.BusinessAcceptsCreditCards',
    'attributes.BikeParking',
    'attributes.RestaurantsTakeOut',
    'attributes.RestaurantsDelivery',
    'attributes.Caters',
    'attributes.WheelchairAccessible',
    'attributes.HappyHour',
    'attributes.OutdoorSeating',
    'attributes.HasTV',
    'attributes.RestaurantsReservations',
    'attributes.DogsAllowed',
    'attributes.GoodForKids',
    'attributes.RestaurantsTableService',
    'attributes.RestaurantsGoodForGroups',
    'attributes.BusinessAcceptsBitcoin'
]
list_attributes = [
    'attributes.WiFi',
    'attributes.Alcohol',
    'attributes.RestaurantsAttire',
    'attributes.NoiseLevel'
]
dict_attributes = [
    'attributes.BusinessParking',
    'attributes.Ambience',
    'attributes.GoodForMeal'
]

In [43]:
pricing_remap = {'1': '$', '2': '$$', '3': '$$$', '4': '$$$$', pd.NA: 'None'}
df_business_cp4[special_attribute] = df_business_cp4[special_attribute].replace(pricing_remap)
df_business_cp4[special_attribute]

0         None
1         None
2           $$
3            $
4         None
          ... 
150341     $$$
150342      $$
150343       $
150344    $$$$
150345       $
Name: attributes.RestaurantsPriceRange2, Length: 150346, dtype: object

In [44]:
df_business_cp4[special_attribute].value_counts()

attributes.RestaurantsPriceRange2
None    65066
$$      48581
$       28840
$$$      6667
$$$$     1192
Name: count, dtype: int64

In [45]:
bool_remap = {'True': True, 'False': False, np.nan: pd.NA, 'None': pd.NA}
for item in bool_attributes:
    df_business_cp4[item] = df_business_cp4[item].replace(bool_remap)
df_business_cp4[bool_attributes] = df_business_cp4[bool_attributes].astype('boolean')

In [46]:

list_remap = {
    np.nan: pd.NA,
    "u'none'": 'None',
    'none': 'None',
    "u'no'": 'No',
    "'no'": 'No',
    "u'free'": 'Free',
    "'free'": 'Free',
    "u'paid'": 'Paid',
    "'paid'": 'Paid',
    "'none'": 'none',
    "u'full_bar'": 'Full Bar',
    "'full_bar'": 'Full Bar',
    "u'beer_and_wine'": 'Beer and Wine',
     "'beer_and_wine'": 'Beer and Wine',
    "u'casual'": 'Casual',
    "'casual'": 'Casual',
    "u'dressy'": 'Dressy',
    "'dressy'": 'Dressy',
    "u'formal'": 'Formal',
    "'formal'": 'Formal',
    "u'quiet'": 'Quiet',
    "'quiet'": 'Quiet',
    "u'average'": 'Average',
    "'average'": 'Average',
    "u'loud'": 'Loud',
    "'loud'": 'Loud',
    "u'very_loud'": 'Very Loud',
    "'very_loud'": 'Very Loud'
}
df_business_cp4[list_attributes] = df_business_cp4[list_attributes].replace(list_remap)
df_business_cp4[list_attributes]

,attributes.WiFi,attributes.Alcohol,attributes.RestaurantsAttire,attributes.NoiseLevel
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,No,<NA>,<NA>,<NA>
3,Free,None,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...
150341,<NA>,<NA>,<NA>,<NA>
150342,No,<NA>,<NA>,<NA>
150343,<NA>,<NA>,<NA>,<NA>
150344,<NA>,<NA>,<NA>,<NA>


In [47]:
for item in list_attributes:
    print(df_business_cp4[item].unique())

[<NA> 'No' 'Free' 'None' 'Paid']
[<NA> 'None' 'Full Bar' 'none' 'Beer and Wine']
[<NA> 'Casual' 'Formal' 'Dressy' 'None']
[<NA> 'Average' 'Quiet' 'Loud' 'Very Loud' 'None']


In [48]:
for col in dict_attributes:
    print(df_business_cp4[col].unique())

[nan
 "{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}"
 "{'garage': False, 'street': True, 'validated': False, 'lot': False, 'valet': False}"
 "{'garage': None, 'street': None, 'validated': None, 'lot': True, 'valet': False}"
 'None'
 "{'garage': False, 'street': False, 'validated': False, 'lot': False, 'valet': False}"
 "{'garage': None, 'street': False, 'validated': None, 'lot': True, 'valet': False}"
 "{u'valet': False, u'garage': None, u'street': True, u'lot': False, u'validated': None}"
 "{'garage': False, 'street': True, 'validated': False, 'lot': True, 'valet': False}"
 "{'garage': True, 'street': False, 'validated': False, 'lot': False, 'valet': False}"
 "{'garage': True, 'street': False, 'validated': True, 'lot': False, 'valet': True}"
 "{'garage': None, 'street': True, 'validated': None, 'lot': False, 'valet': False}"
 "{'garage': False, 'street': True, 'validated': True, 'lot': True, 'valet': False}"
 "{u'valet': False, u'garage': False, 

### Creating backup

In [49]:
df_business_cp5 = df_business_cp4.copy()

### Cleaning up "Open" column to True/False instead of 1/0

In [50]:
df_business_cp5['is_open'] = df_business_cp5['is_open'].astype('bool')

### Creating Open and Closed dates from the review dataset
Dependency on yelp_review dataset to be cleaned

In [51]:
path_review = '/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_review_CLEAN.csv'

In [52]:
df_review = lib.read_data(path_review)

PortfolioLogger.lib.tools: INFO: Encoding: ascii
PortfolioLogger.lib.tools: INFO: Stripping whitespaces from yelp_review_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function read_data at 0x10f9a1800> took 2.075 mins to complete.


In [53]:
df_review

,Unnamed: 0,review_id,user_id,business_id,stars,useful,funny,cool,date
0,0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,2018-07-07 22:09:11
1,1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,1,0,1,2012-01-03 15:28:18
2,2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,2014-02-05 20:30:30
3,3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,1,0,1,2015-01-04 00:01:03
4,4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,1,0,1,2017-01-14 20:54:15
...,...,...,...,...,...,...,...,...,...
6990275,6990275,H0RIamZu0B0Ei0P4aeh3sQ,qskILQ3k0I_qcCMI-k6_QQ,jals67o91gcrD4DC81Vk6w,5.0,1,2,1,2014-12-17 21:45:20
6990276,6990276,shTPgbgdwTHSuU67mGCmZQ,Zo0th2m8Ez4gLSbHftiQvg,2vLksaMmSEcGbjI5gywpZA,5.0,2,1,2,2021-03-31 16:55:10
6990277,6990277,YNfNhgZlaaCO5Q_YJR4rEw,mm6E4FbCMwJmb7kPDZ5v2Q,R1khUUxidqfaJmcpmGd4aw,4.0,1,0,0,2019-12-30 03:56:30
6990278,6990278,i-I4ZOhoX70Nw5H0FwrQUA,YwAMC-jvZ1fvEUum6QkEkw,Rr9kKArrMhSLVE9a53q-aA,5.0,1,0,0,2022-01-19 18:59:27


In [54]:
df_review

,Unnamed: 0,review_id,user_id,business_id,stars,useful,funny,cool,date
0,0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,2018-07-07 22:09:11
1,1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,1,0,1,2012-01-03 15:28:18
2,2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,2014-02-05 20:30:30
3,3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,1,0,1,2015-01-04 00:01:03
4,4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,1,0,1,2017-01-14 20:54:15
...,...,...,...,...,...,...,...,...,...
6990275,6990275,H0RIamZu0B0Ei0P4aeh3sQ,qskILQ3k0I_qcCMI-k6_QQ,jals67o91gcrD4DC81Vk6w,5.0,1,2,1,2014-12-17 21:45:20
6990276,6990276,shTPgbgdwTHSuU67mGCmZQ,Zo0th2m8Ez4gLSbHftiQvg,2vLksaMmSEcGbjI5gywpZA,5.0,2,1,2,2021-03-31 16:55:10
6990277,6990277,YNfNhgZlaaCO5Q_YJR4rEw,mm6E4FbCMwJmb7kPDZ5v2Q,R1khUUxidqfaJmcpmGd4aw,4.0,1,0,0,2019-12-30 03:56:30
6990278,6990278,i-I4ZOhoX70Nw5H0FwrQUA,YwAMC-jvZ1fvEUum6QkEkw,Rr9kKArrMhSLVE9a53q-aA,5.0,1,0,0,2022-01-19 18:59:27


In [55]:
df_business_cp5.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'categories', 'attributes.ByAppointmentOnly',
       'attributes.BusinessAcceptsCreditCards', 'attributes.BikeParking',
       'attributes.RestaurantsPriceRange2', 'attributes.RestaurantsTakeOut',
       'attributes.RestaurantsDelivery', 'attributes.Caters',
       'attributes.WiFi', 'attributes.BusinessParking',
       'attributes.WheelchairAccessible', 'attributes.HappyHour',
       'attributes.OutdoorSeating', 'attributes.HasTV',
       'attributes.RestaurantsReservations', 'attributes.DogsAllowed',
       'attributes.Alcohol', 'attributes.GoodForKids',
       'attributes.RestaurantsAttire', 'attributes.Ambience',
       'attributes.RestaurantsTableService',
       'attributes.RestaurantsGoodForGroups', 'attributes.NoiseLevel',
       'attributes.GoodForMeal', 'attributes.BusinessAcceptsBitcoin'],
      dtype='object')

In [56]:
df_review = df_review[['business_id', 'date']]

In [57]:
df_review

,business_id,date
0,XQfwVwDr-v0ZS3_CbbE5Xw,2018-07-07 22:09:11
1,7ATYjTIgM3jUlt4UM3IypQ,2012-01-03 15:28:18
2,YjUWPpI6HXG530lwP-fb2A,2014-02-05 20:30:30
3,kxX2SOes4o-D3ZQBkiMRfA,2015-01-04 00:01:03
4,e4Vwtrqf-wpJfwesgvdgxQ,2017-01-14 20:54:15
...,...,...
6990275,jals67o91gcrD4DC81Vk6w,2014-12-17 21:45:20
6990276,2vLksaMmSEcGbjI5gywpZA,2021-03-31 16:55:10
6990277,R1khUUxidqfaJmcpmGd4aw,2019-12-30 03:56:30
6990278,Rr9kKArrMhSLVE9a53q-aA,2022-01-19 18:59:27


In [58]:
review_ranges = df_review.groupby("business_id")['date'].agg(
    open_date='min',
    close_date='max'
).reset_index()

merged = df_business_cp5[['business_id', 'is_open']].merge(
    review_ranges,
    on='business_id',
    how='left'
)

# Assign NaT to businesses that are still open
merged['close_date'] = merged.apply(
    lambda r: pd.NaT if r['is_open'] == 1 else r['close_date'],
    axis=1
)

In [59]:
df_business_cp5 = df_business_cp5.merge(merged[['business_id', 'open_date', 'close_date']], on='business_id', how='left')

In [60]:
df_business_cp5['open_date'] = pd.to_datetime(df_business_cp5['open_date'])
df_business_cp5['close_date'] = pd.to_datetime(df_business_cp5['close_date'])

In [61]:
columns = list(df_business_cp5.columns)
columns

['business_id',
 'name',
 'address',
 'city',
 'state',
 'postal_code',
 'latitude',
 'longitude',
 'stars',
 'review_count',
 'is_open',
 'categories',
 'attributes.ByAppointmentOnly',
 'attributes.BusinessAcceptsCreditCards',
 'attributes.BikeParking',
 'attributes.RestaurantsPriceRange2',
 'attributes.RestaurantsTakeOut',
 'attributes.RestaurantsDelivery',
 'attributes.Caters',
 'attributes.WiFi',
 'attributes.BusinessParking',
 'attributes.WheelchairAccessible',
 'attributes.HappyHour',
 'attributes.OutdoorSeating',
 'attributes.HasTV',
 'attributes.RestaurantsReservations',
 'attributes.DogsAllowed',
 'attributes.Alcohol',
 'attributes.GoodForKids',
 'attributes.RestaurantsAttire',
 'attributes.Ambience',
 'attributes.RestaurantsTableService',
 'attributes.RestaurantsGoodForGroups',
 'attributes.NoiseLevel',
 'attributes.GoodForMeal',
 'attributes.BusinessAcceptsBitcoin',
 'open_date',
 'close_date']

In [62]:
reordered_columns = ['business_id',
 'name',
 'address',
 'city',
 'state',
 'postal_code',
 'latitude',
 'longitude',
 'stars',
 'review_count',
 'is_open',
 'open_date',
 'close_date',
 'categories',
 'attributes.ByAppointmentOnly',
 'attributes.BusinessAcceptsCreditCards',
 'attributes.BikeParking',
 'attributes.RestaurantsPriceRange2',
 'attributes.RestaurantsTakeOut',
 'attributes.RestaurantsDelivery',
 'attributes.Caters',
 'attributes.WiFi',
 'attributes.BusinessParking',
 'attributes.WheelchairAccessible',
 'attributes.HappyHour',
 'attributes.OutdoorSeating',
 'attributes.HasTV',
 'attributes.RestaurantsReservations',
 'attributes.DogsAllowed',
 'attributes.Alcohol',
 'attributes.GoodForKids',
 'attributes.RestaurantsAttire',
 'attributes.Ambience',
 'attributes.RestaurantsTableService',
 'attributes.RestaurantsGoodForGroups',
 'attributes.NoiseLevel',
 'attributes.GoodForMeal',
 'attributes.BusinessAcceptsBitcoin',]

In [63]:
df_business_cp5 = df_business_cp5[reordered_columns]

In [64]:
df_business_cp5

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,attributes.DogsAllowed,attributes.Alcohol,attributes.GoodForKids,attributes.RestaurantsAttire,attributes.Ambience,attributes.RestaurantsTableService,attributes.RestaurantsGoodForGroups,attributes.NoiseLevel,attributes.GoodForMeal,attributes.BusinessAcceptsBitcoin
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,...,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,NaN,<NA>
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,...,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,NaN,<NA>
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,...,False,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,NaN,<NA>
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,...,<NA>,None,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,NaN,<NA>
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,...,<NA>,<NA>,True,<NA>,NaN,<NA>,<NA>,<NA>,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150341,IUQopTMmYQG-qRtBk-8QnA,Binh's Nails,3388 Gateway Blvd,Edmonton,AB,T6J 5H2,53.468419,-113.492054,3.0,13,...,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,NaN,<NA>
150342,c8GjPIOTGVmIemT7j5_SyQ,Wild Birds Unlimited,2813 Bransford Ave,Nashville,TN,37204,36.115118,-86.766925,4.0,5,...,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,NaN,<NA>
150343,_QAMST-NrQobXduilWEqSw,Claire's Boutique,"6020 E 82nd St, Ste 46",Indianapolis,IN,46250,39.908707,-86.065088,3.5,8,...,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,NaN,<NA>
150344,mtGm22y5c2UHNXDFAjaPNw,Cyclery & Fitness Center,2472 Troy Rd,Edwardsville,IL,62025,38.782351,-89.950558,4.0,24,...,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,NaN,<NA>


In [65]:
df_business_cp5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 38 columns):
 #   Column                                 Non-Null Count   Dtype         
---  ------                                 --------------   -----         
 0   business_id                            150346 non-null  object        
 1   name                                   150346 non-null  object        
 2   address                                150346 non-null  object        
 3   city                                   150346 non-null  object        
 4   state                                  150346 non-null  object        
 5   postal_code                            150346 non-null  object        
 6   latitude                               150346 non-null  float64       
 7   longitude                              150346 non-null  float64       
 8   stars                                  150346 non-null  float64       
 9   review_count                           150346 no

## Cleaning States column
### Creating a backup

In [35]:
df_business_cp6 = df_business_cp5.copy()

## So I don't have to reprocess the whole dataset

In [3]:
# df_business_cp6 = lib.read_data(export_clean_directory / 'yelp_business_CLEAN.csv')

PortfolioLogger.lib.tools: INFO: Encoding: utf-8
PortfolioLogger.lib.tools: INFO: Stripping whitespaces from yelp_business_CLEAN.csv
PortfolioLogger.lib.tools: ERROR: Could not strip whitespaces for attributes.ByAppointmentOnly:
Can only use .str accessor with string values!
PortfolioLogger.lib.tools: ERROR: Could not strip whitespaces for attributes.BusinessAcceptsCreditCards:
Can only use .str accessor with string values!
PortfolioLogger.lib.tools: ERROR: Could not strip whitespaces for attributes.BikeParking:
Can only use .str accessor with string values!
PortfolioLogger.lib.tools: ERROR: Could not strip whitespaces for attributes.RestaurantsTakeOut:
Can only use .str accessor with string values!
PortfolioLogger.lib.tools: ERROR: Could not strip whitespaces for attributes.RestaurantsDelivery:
Can only use .str accessor with string values!
PortfolioLogger.lib.tools: ERROR: Could not strip whitespaces for attributes.Caters:
Can only use .str accessor with string values!
PortfolioLogge

In [36]:
df_business_cp6

,Unnamed: 0,business_id,name,address,city,state,postal_code,latitude,longitude,stars,...,attributes.DogsAllowed,attributes.Alcohol,attributes.GoodForKids,attributes.RestaurantsAttire,attributes.Ambience,attributes.RestaurantsTableService,attributes.RestaurantsGoodForGroups,attributes.NoiseLevel,attributes.GoodForMeal,attributes.BusinessAcceptsBitcoin
0,0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,...,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150341,150341,IUQopTMmYQG-qRtBk-8QnA,Binh's Nails,3388 Gateway Blvd,Edmonton,AB,T6J 5H2,53.468419,-113.492054,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150342,150342,c8GjPIOTGVmIemT7j5_SyQ,Wild Birds Unlimited,2813 Bransford Ave,Nashville,TN,37204,36.115118,-86.766925,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150343,150343,_QAMST-NrQobXduilWEqSw,Claire's Boutique,"6020 E 82nd St, Ste 46",Indianapolis,IN,46250,39.908707,-86.065088,3.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150344,150344,mtGm22y5c2UHNXDFAjaPNw,Cyclery & Fitness Center,2472 Troy Rd,Edwardsville,IL,62025,38.782351,-89.950558,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
df_business_cp6.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 39 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   Unnamed: 0                             150346 non-null  int64  
 1   business_id                            150346 non-null  object 
 2   name                                   150346 non-null  object 
 3   address                                145219 non-null  object 
 4   city                                   150346 non-null  object 
 5   state                                  150346 non-null  object 
 6   postal_code                            150273 non-null  object 
 7   latitude                               150346 non-null  float64
 8   longitude                              150346 non-null  float64
 9   stars                                  150346 non-null  float64
 10  review_count                           150346 non-null  

In [38]:
state_group = df_business_cp6.groupby('state')

In [39]:
states_to_drop = []
for state, data in state_group:
    if len(data['business_id'].unique()) <= 1000:
        states_to_drop.append(state)
    else:
        log.info(f'Valid State: {state}')
states_to_drop

PortfolioLogger.yelp_dataset: INFO: Valid State: AB
PortfolioLogger.yelp_dataset: INFO: Valid State: AZ
PortfolioLogger.yelp_dataset: INFO: Valid State: CA
PortfolioLogger.yelp_dataset: INFO: Valid State: DE
PortfolioLogger.yelp_dataset: INFO: Valid State: FL
PortfolioLogger.yelp_dataset: INFO: Valid State: ID
PortfolioLogger.yelp_dataset: INFO: Valid State: IL
PortfolioLogger.yelp_dataset: INFO: Valid State: IN
PortfolioLogger.yelp_dataset: INFO: Valid State: LA
PortfolioLogger.yelp_dataset: INFO: Valid State: MO
PortfolioLogger.yelp_dataset: INFO: Valid State: NJ
PortfolioLogger.yelp_dataset: INFO: Valid State: NV
PortfolioLogger.yelp_dataset: INFO: Valid State: PA
PortfolioLogger.yelp_dataset: INFO: Valid State: TN


['CO', 'HI', 'MA', 'MI', 'MT', 'NC', 'SD', 'TX', 'UT', 'VI', 'VT', 'WA', 'XMS']

In [41]:
indicies_to_drop = df_business_cp6[df_business_cp6['state'].isin(states_to_drop)].index
indicies_to_drop

Index([  9101,   9126,  10948,  12258,  14416,  29606,  32345,  32536,  34661,
        44088,  48919,  57832,  76352,  88413,  95091,  97813,  98685, 125353,
       126989, 135715, 144012],
      dtype='int64')

In [45]:
df_business_cp6 = df_business_cp6.drop(indicies_to_drop)

In [49]:
df_business_cp6.drop('Unnamed: 0', axis=1, inplace=True)
df_business_cp6

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,attributes.DogsAllowed,attributes.Alcohol,attributes.GoodForKids,attributes.RestaurantsAttire,attributes.Ambience,attributes.RestaurantsTableService,attributes.RestaurantsGoodForGroups,attributes.NoiseLevel,attributes.GoodForMeal,attributes.BusinessAcceptsBitcoin
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,...,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150341,IUQopTMmYQG-qRtBk-8QnA,Binh's Nails,3388 Gateway Blvd,Edmonton,AB,T6J 5H2,53.468419,-113.492054,3.0,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150342,c8GjPIOTGVmIemT7j5_SyQ,Wild Birds Unlimited,2813 Bransford Ave,Nashville,TN,37204,36.115118,-86.766925,4.0,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150343,_QAMST-NrQobXduilWEqSw,Claire's Boutique,"6020 E 82nd St, Ste 46",Indianapolis,IN,46250,39.908707,-86.065088,3.5,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150344,mtGm22y5c2UHNXDFAjaPNw,Cyclery & Fitness Center,2472 Troy Rd,Edwardsville,IL,62025,38.782351,-89.950558,4.0,24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Get state with the most businesses/reviews to analyze

In [79]:
state_group = df_business_cp6.groupby('state', sort=True)
state_group

In [121]:
print(state_group['business_id'].size().max())
print(state_group['business_id'].size())
print(state_group['review_count'].sum().max())
print(state_group['review_count'].sum())

# state_group.filter(state_group['business_id'].size() == state_group['business_id'].size().max())
# state_group['business_id'].transform('size')
state_group['business_id'].size() == state_group['business_id'].size().max()
state_group['review_count'].sum() == state_group['review_count'].sum().max()

34039
state
AB     5573
AZ     9912
CA     5203
DE     2265
FL    26330
ID     4467
IL     2145
IN    11247
LA     9924
MO    10913
NJ     8536
NV     7715
PA    34039
TN    12056
Name: business_id, dtype: int64
1540790
state
AB     105477
AZ     412639
CA     339637
DE      67370
FL    1119926
ID     152086
IL      49676
IN     472565
LA     743176
MO     483897
NJ     249837
NV     409950
PA    1540790
TN     598195
Name: review_count, dtype: int64


state
AB    False
AZ    False
CA    False
DE    False
FL    False
ID    False
IL    False
IN    False
LA    False
MO    False
NJ    False
NV    False
PA     True
TN    False
Name: review_count, dtype: bool

In [123]:
df_business_cp_PA = df_business_cp6[df_business_cp6['state'] == 'PA']

In [124]:
lib.export_data(EXPORT_DIRECTORY / 'yelp_business_PA.csv', df_business_cp_PA)

PortfolioLogger.lib.tools: INFO: Exporting 1293482 elements (51.50 MB) to /Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_business_PA_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10dba1940> took 0.268 secs to complete.


PosixPath('/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_business_PA_CLEAN.csv')

# Exporting Cleaned Data
## Creating Clean backup

In [50]:
df_business_CLEANED = df_business_cp6.copy()

In [47]:
lib.export_data(EXPORT_DIRECTORY / 'yelp_business.csv', df_business_CLEANED)

PortfolioLogger.lib.tools: INFO: Exporting 5862675 elements (227.86 MB) to /Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_business_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10dba1940> took 1.102 secs to complete.


PosixPath('/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/yelp/yelp_dataset/clean/yelp_business_CLEAN.csv')